In [2]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from IPython.display import Markdown, display
import sqlite3

# MODEL = 'gpt-4.1-mini'
# openai = OpenAI()



MODEL = 'llama3.2:1b'
openai = OpenAI(base_url='http://localhost:11434/v1',api_key='ollama')

In [3]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}


set_price_function = {
    "name": "set_ticket_price",
    "description": "update or set the price of ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city where the price needs to be updated or added.",
            },
            "price":
            {
                "type": "number",
                "description": "The new ticket price amount as a number",
            },
        },
        "required": ["destination_city", "price"],
        "additionalProperties": False
    }
}

In [4]:
tools = [{"type": "function", "function": price_function},{"type": "function", "function": set_price_function}]

In [6]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
        if tool_call.function.name == "set_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price = arguments.get('price')
            price_details = set_ticket_price(city,price)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
# ticket_prices = {"london": "$789", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

# def get_ticket_price(destination_city):
#     print(f"Tool called for city {destination_city}")
#     price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
#     return f"The price of a ticket to {destination_city} is {price}"

# def set_ticket_price(city, price_value):
#     cleaned_city = city.strip().lower()
#     if isinstance(price_value, str) and "$" in price_value:
#         price_value = price_value.replace("$", "").strip()
#         final_price = int(price_value)
#         ticket_prices[cleaned_city] = f"${final_price}"
#     return f"The price of a ticket to {city} is {price_value}"

# def set_ticket_price(city, price):
#     with sqlite3.connect(DB) as conn:
#         cursor = conn.cursor()
#         cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
#         conn.commit()


In [ ]:
DB = "prices.db"
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

# get_ticket_price("london")


In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""
def chat(message, history):
    history = [{"role":h['role'], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    # for i in messages:
    #     print(i)
    # print(response)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        # print(message)
        responses = handle_tool_calls(message)
        # print("response",response)
        messages.append(message)
        # print("\n",messages)
        messages.extend(responses)
        # print("\n",messages)
        response = openai.chat.completions.create(model=MODEL, messages=messages,tools=tools)

    return response.choices[0].message.content

In [ ]:
# Check the price for London. If the price is under $500, check the price for Sydney. But if the price for London is $500 or more, check the price for Tokyo instead. Only give me the price of the final city you checked with city name.

# ticket_prices = {"london": "$789", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}